# 🍽️ Cantina Escolar — Análise de Dados e Big Data (Camada Gold + Modelos)

**Projeto Integrador — Grupo 3 (Data Science)**

Este notebook parte dos **dados já limpos** (camada **Silver**) da cantina e entrega:

| Etapa | O que faz | Conteúdo da disciplina usado |
|---|---|---|
| 1. Carregar | Lê as 8 tabelas limpas e confere se as colunas estão certas | — |
| 2. Juntar | Cruza vendas com cardápio, alunos, calendário e clima | — |
| 3. Camada Gold | Gera as 5 tabelas analíticas pedidas na especificação (+ inflação) | — |
| 4. Hadoop | **MapReduce** (contagem distribuída), **Hive** (consultas SQL) e **Sqoop** (exportação para banco) | Aula 08 |
| 5. Regressão | Prevê **quantos itens a cantina vai vender** em um dia, e sugere a produção | Aula 06 |
| 6. Classificação | Prevê se uma **compra fiado vai atrasar** o pagamento | Aula 05 |
| 7. Spark | **Spark SQL** e **Structured Streaming** simulando a chegada das vendas dia a dia | Aula 09 |
| 8. Resumo | Principais números e exportação dos resultados | — |

> **Como usar:** `Ambiente de execução → Executar tudo`. A única coisa a ajustar é a célula de
> **configuração** (seção 1), dizendo onde estão os dados limpos.

**O que ficou de fora das aulas (e por quê):** as simulações de **YARN**, **Pig** e **HBase** da Aula 08
não acrescentam nada a este projeto — o volume de dados da cantina cabe na memória, e a mesma ideia de
"fluxo de transformação" (Pig) e "consulta" (Hive) já está coberta pelo pandas, pelo SQL e pelo Spark.
A coleta de uma API externa da Aula 09 também não se aplica: aqui os "dados ao vivo" são as próprias
vendas da cantina.

## 0. Instalação e bibliotecas

O Colab já vem com quase tudo. Só garantimos o **PySpark** (usado na seção 7).

In [ ]:
%pip -q install pyspark

In [ ]:
import os
import re
import json
import shutil
import sqlite3
import subprocess
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# Visual dos gráficos: grade discreta e cores categóricas em ordem fixa
sns.set_theme(style="whitegrid")
CORES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
COR = CORES[0]
plt.rcParams.update({"figure.figsize": (10, 4.5), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold"})

DIAS_SEMANA = {0: "Seg", 1: "Ter", 2: "Qua", 3: "Qui", 4: "Sex", 5: "Sáb", 6: "Dom"}
ORDEM_DIAS = list(DIAS_SEMANA.values())
print("Bibliotecas carregadas.")

## 1. Configuração e carregamento dos dados limpos

Os dados limpos (camada **Silver**) devem estar em uma pasta com **um arquivo por tabela**.
Cada arquivo pode ser `.csv`, `.parquet` ou `.xlsx`, com estes nomes e colunas:

| Arquivo | Colunas obrigatórias |
|---|---|
| `vendas` | id_venda, data_hora, id_aluno, id_produto, quantidade, valor_unitario, valor_total, forma_pagamento |
| `devedores` | id_devedor, id_aluno, data_divida, valor_devido, status, data_quitacao |
| `cardapio` | id_produto, nome_produto, categoria, preco_venda, custo_unitario |
| `perdas` | id_perda, data, id_produto, quantidade, motivo, valor_prejuizo |
| `calendario` | data, tipo_dia, descricao, turno_letivo |
| `clima` | data, temperatura_media, temperatura_max, condicao, precipitacao_mm |
| `alunos` | turma, ano_letivo, turno, id_aluno, nome, data_nascimento, restricao_alimentar |
| `inflacao` | mes_referencia, indice_inflacao, categoria_produto |

O que se espera da limpeza (feita em outra etapa): **IDs no mesmo formato em todas as tabelas**
(`id_aluno` e `id_produto` numéricos), datas em `aaaa-mm-dd`, sem duplicatas, sem valores negativos
ou absurdos, `status` em *Quitado / Em aberto / Atrasado* e categorias da inflação com os mesmos nomes
do cardápio. Colunas extras são permitidas e simplesmente ignoradas.

**Escolha a origem** na célula abaixo:
- `"github"` → baixa o repositório do projeto e lê `dados/silver/` (padrão);
- `"drive"` → lê de uma pasta do seu Google Drive.

In [ ]:
# ===================== CONFIGURAÇÃO (ajuste se necessário) =====================
ORIGEM_DADOS = "github"          # "github" ou "drive"

REPO_URL = "https://github.com/PedroFreitasDev/P.I-Data-Science.git"
PASTA_REPO = "/content/P.I-Data-Science"
SUBPASTA_LIMPOS = "dados/silver"  # pasta dos dados limpos dentro do repositório

PASTA_DRIVE = "/content/drive/MyDrive/P.I-Data-Science/dados/silver"

PASTA_SAIDA = "/content/resultados"   # onde as tabelas Gold e o banco serão salvos
# ===============================================================================

if ORIGEM_DADOS == "github":
    if os.path.isdir(os.path.join(PASTA_REPO, ".git")):
        subprocess.run(["git", "-C", PASTA_REPO, "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, PASTA_REPO], check=True)
    PASTA_DADOS = os.path.join(PASTA_REPO, SUBPASTA_LIMPOS)
elif ORIGEM_DADOS == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    PASTA_DADOS = PASTA_DRIVE
else:
    raise ValueError('ORIGEM_DADOS deve ser "github" ou "drive"')

os.makedirs(PASTA_SAIDA, exist_ok=True)
print("Lendo os dados limpos de:", PASTA_DADOS)

In [ ]:
# "Contrato" das tabelas limpas: nome do arquivo -> colunas obrigatórias
CONTRATO = {
    "vendas":     ["id_venda", "data_hora", "id_aluno", "id_produto", "quantidade",
                   "valor_unitario", "valor_total", "forma_pagamento"],
    "devedores":  ["id_devedor", "id_aluno", "data_divida", "valor_devido", "status", "data_quitacao"],
    "cardapio":   ["id_produto", "nome_produto", "categoria", "preco_venda", "custo_unitario"],
    "perdas":     ["id_perda", "data", "id_produto", "quantidade", "motivo", "valor_prejuizo"],
    "calendario": ["data", "tipo_dia", "descricao", "turno_letivo"],
    "clima":      ["data", "temperatura_media", "temperatura_max", "condicao", "precipitacao_mm"],
    "alunos":     ["turma", "ano_letivo", "turno", "id_aluno", "nome", "data_nascimento",
                   "restricao_alimentar"],
    "inflacao":   ["mes_referencia", "indice_inflacao", "categoria_produto"],
}
LEITORES = {".parquet": pd.read_parquet, ".csv": pd.read_csv, ".xlsx": pd.read_excel}


def carregar(nome):
    # Procura o arquivo da tabela em qualquer um dos formatos aceitos
    for extensao, leitor in LEITORES.items():
        caminho = os.path.join(PASTA_DADOS, nome + extensao)
        if os.path.exists(caminho):
            df = leitor(caminho)
            faltando = [c for c in CONTRATO[nome] if c not in df.columns]
            if faltando:
                raise ValueError(f"'{nome}{extensao}' está sem as colunas obrigatórias: {faltando}")
            return df
    raise FileNotFoundError(f"Não encontrei '{nome}' (.parquet, .csv ou .xlsx) em {PASTA_DADOS}")


dados = {nome: carregar(nome) for nome in CONTRATO}
display(pd.DataFrame({"tabela": list(dados),
                      "linhas": [len(d) for d in dados.values()],
                      "colunas": [d.shape[1] for d in dados.values()]}))

### 1.1 Acertando os tipos

Mesmo com os dados limpos, ao ler um CSV as datas chegam como texto. Aqui convertemos datas,
e criamos colunas auxiliares nos alunos: **série** (1 a 12), **segmento** (Fundamental I, Fundamental II,
Médio) e **idade**.

In [ ]:
vendas = dados["vendas"].copy()
devedores = dados["devedores"].copy()
cardapio = dados["cardapio"].copy()
perdas = dados["perdas"].copy()
calendario = dados["calendario"].copy()
clima = dados["clima"].copy()
alunos = dados["alunos"].copy()
inflacao = dados["inflacao"].copy()

vendas["data_hora"] = pd.to_datetime(vendas["data_hora"])
devedores["data_divida"] = pd.to_datetime(devedores["data_divida"])
devedores["data_quitacao"] = pd.to_datetime(devedores["data_quitacao"], errors="coerce")
perdas["data"] = pd.to_datetime(perdas["data"])
calendario["data"] = pd.to_datetime(calendario["data"])
clima["data"] = pd.to_datetime(clima["data"])
alunos["data_nascimento"] = pd.to_datetime(alunos["data_nascimento"])
inflacao["mes_referencia"] = pd.to_datetime(inflacao["mes_referencia"].astype(str)).dt.to_period("M")

# O cardápio pode ter mais de uma versão (1º e 2º semestre): fica a mais recente de cada produto
cardapio = cardapio.drop_duplicates("id_produto", keep="last")


def serie_da_turma(turma):
    # "7º ano A" -> 7 ; "2ª série EM B" -> 11 (Ensino Médio vem depois do 9º ano)
    numero = int(re.match(r"\s*(\d+)", str(turma)).group(1))
    return numero + 9 if "EM" in str(turma) else numero


alunos["serie"] = alunos["turma"].apply(serie_da_turma)
alunos["segmento"] = pd.cut(alunos["serie"], [0, 5, 9, 12],
                            labels=["Fundamental I", "Fundamental II", "Médio"]).astype(str)
alunos["idade"] = alunos["ano_letivo"] - alunos["data_nascimento"].dt.year

# Ordem "natural" das turmas (1º ano A, 1º ano B, ..., 3ª série EM B) para os gráficos
ORDEM_TURMAS = alunos.sort_values(["serie", "turma"])["turma"].unique().tolist()

print(f"Período das vendas: {vendas['data_hora'].min():%d/%m/%Y} a {vendas['data_hora'].max():%d/%m/%Y}")
print(f"{alunos['id_aluno'].nunique()} alunos em {len(ORDEM_TURMAS)} turmas, "
      f"{cardapio['id_produto'].nunique()} produtos no cardápio")

## 2. Juntando as tabelas (enriquecimento)

Uma venda sozinha diz pouco: "aluno 1234 comprou o produto 7". Juntando com as outras tabelas,
ela passa a dizer **"um aluno do 7º ano A comprou um misto quente (Lanches Naturais) numa
terça-feira fria e chuvosa"**. É isso que permite responder às perguntas de negócio.

Também montamos uma tabela com **um registro por dia de aula** (`dias`), com o total vendido,
o clima e o tipo de dia — ela é a base da relação clima × vendas e do modelo de regressão.

> Os IDs de alunos recomeçam a cada ano letivo, por isso a junção com alunos usa **id_aluno + ano**.

In [ ]:
vendas["data"] = vendas["data_hora"].dt.normalize()
vendas["ano_letivo"] = vendas["data_hora"].dt.year
vendas["mes"] = vendas["data_hora"].dt.to_period("M")
vendas["hora"] = vendas["data_hora"].dt.hour
vendas["dia_semana"] = vendas["data"].dt.dayofweek.map(DIAS_SEMANA)

ve = (vendas
      .merge(cardapio[["id_produto", "nome_produto", "categoria", "custo_unitario"]],
             on="id_produto", how="left")
      .merge(alunos[["id_aluno", "ano_letivo", "turma", "serie", "segmento", "turno"]],
             on=["id_aluno", "ano_letivo"], how="left")
      .merge(clima, on="data", how="left"))
ve["lucro"] = ve["valor_total"] - ve["quantidade"] * ve["custo_unitario"]

# Um registro por dia com vendas
vendas_dia = (ve.groupby("data")
                .agg(n_vendas=("id_venda", "nunique"),
                     itens_vendidos=("quantidade", "sum"),
                     faturamento=("valor_total", "sum"))
                .reset_index())

dias = (vendas_dia
        .merge(calendario, on="data", how="left")
        .merge(clima, on="data", how="left"))
dias["dia_semana"] = dias["data"].dt.dayofweek.map(DIAS_SEMANA)
dias["mes"] = dias["data"].dt.month
dias["dia_do_mes"] = dias["data"].dt.day

print(f"Vendas enriquecidas: {len(ve):,} linhas | Dias com vendas: {len(dias)}")
sem_turma = ve["turma"].isna().mean()
sem_produto = ve["categoria"].isna().mean()
print(f"Vendas sem turma encontrada: {sem_turma:.1%} | sem produto no cardápio: {sem_produto:.1%}")
display(ve.head())

### 2.1 Primeira olhada (análise exploratória)

In [ ]:
print("Resumo dos dias de aula:")
display(dias[["n_vendas", "itens_vendidos", "faturamento", "temperatura_media", "precipitacao_mm"]]
        .describe().T)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(dias["data"], dias["faturamento"], color=COR, linewidth=2)
ax.set_title("Faturamento por dia de aula")
ax.set_ylabel("R$")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
(ve.groupby("forma_pagamento")["valor_total"].sum().sort_values()
   .plot.barh(ax=axes[0], color=COR, title="Faturamento por forma de pagamento (R$)"))
(ve.groupby("hora")["id_venda"].nunique()
   .plot.bar(ax=axes[1], color=COR, title="Número de vendas por hora do dia", rot=0))
axes[0].set_ylabel("")
axes[1].set_xlabel("hora")
plt.tight_layout()
plt.show()

## 3. Camada Gold — tabelas prontas para análise

A especificação do projeto pede estas tabelas agregadas:

1. **Venda por turma/mês**
2. **Inadimplência por período/turma**
3. **Relação clima × volume de vendas**
4. **Desperdício por categoria de produto**
5. **Turmas × dias de pico de venda**

E, como bônus, **reajuste de preços × inflação** (usa a fonte de inflação, que de outra forma ficaria sem uso).
Todas são guardadas no dicionário `gold` e exportadas no final.

In [ ]:
gold = {}

# ---------------- 3.1 Venda por turma/mês ----------------
gold["venda_turma_mes"] = (ve.dropna(subset=["turma"])
    .groupby(["turma", "mes"])
    .agg(n_vendas=("id_venda", "nunique"),
         itens=("quantidade", "sum"),
         faturamento=("valor_total", "sum"),
         lucro=("lucro", "sum"))
    .reset_index())
gold["venda_turma_mes"]["ticket_medio"] = (gold["venda_turma_mes"]["faturamento"]
                                           / gold["venda_turma_mes"]["n_vendas"])
display(gold["venda_turma_mes"].head(10))

mapa = (gold["venda_turma_mes"]
        .pivot(index="turma", columns="mes", values="faturamento")
        .reindex([t for t in ORDEM_TURMAS if t in set(gold["venda_turma_mes"]["turma"])]))
plt.figure(figsize=(12, 7))
sns.heatmap(mapa, cmap="Blues", linewidths=0.5, cbar_kws={"label": "Faturamento (R$)"})
plt.title("Faturamento por turma e mês (quanto mais escuro, mais vendeu)")
plt.xlabel("")
plt.ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# ---------------- 3.2 Inadimplência por período/turma ----------------
STATUS_ATRASADO, STATUS_ABERTO, STATUS_QUITADO = "Atrasado", "Em aberto", "Quitado"

dev = devedores.copy()
dev["ano_letivo"] = dev["data_divida"].dt.year
dev["mes"] = dev["data_divida"].dt.to_period("M")
dev = dev.merge(alunos[["id_aluno", "ano_letivo", "turma", "serie", "segmento", "turno",
                        "idade", "restricao_alimentar"]],
                on=["id_aluno", "ano_letivo"], how="left")
dev["valor_atrasado"] = dev["valor_devido"].where(dev["status"] == STATUS_ATRASADO, 0)
dev["valor_em_aberto"] = dev["valor_devido"].where(dev["status"] == STATUS_ABERTO, 0)

gold["inadimplencia_turma_mes"] = (dev.dropna(subset=["turma"])
    .groupby(["turma", "mes"])
    .agg(n_dividas=("id_devedor", "count"),
         valor_total=("valor_devido", "sum"),
         valor_atrasado=("valor_atrasado", "sum"),
         valor_em_aberto=("valor_em_aberto", "sum"))
    .reset_index())
gold["inadimplencia_turma_mes"]["pct_atrasado"] = (100 * gold["inadimplencia_turma_mes"]["valor_atrasado"]
                                                   / gold["inadimplencia_turma_mes"]["valor_total"])
display(gold["inadimplencia_turma_mes"].head(10))

por_turma = (dev.dropna(subset=["turma"]).groupby("turma")[["valor_devido", "valor_atrasado"]].sum())
por_turma["pct_atrasado"] = 100 * por_turma["valor_atrasado"] / por_turma["valor_devido"]
por_turma = por_turma.reindex([t for t in ORDEM_TURMAS if t in por_turma.index])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
por_turma["pct_atrasado"].plot.barh(ax=axes[0], color=COR)
axes[0].invert_yaxis()
axes[0].set_title("% do valor fiado que está atrasado, por turma")
axes[0].set_xlabel("%")
axes[0].set_ylabel("")
mensal = dev.groupby("mes")[["valor_devido", "valor_atrasado"]].sum()
axes[1].plot(mensal.index.astype(str), mensal["valor_devido"], color=CORES[0], linewidth=2, marker="o",
             label="Total vendido fiado")
axes[1].plot(mensal.index.astype(str), mensal["valor_atrasado"], color=CORES[1], linewidth=2, marker="o",
             label="Atrasado")
axes[1].set_title("Fiado por mês (R$)")
axes[1].legend()
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------- 3.3 Relação clima × volume de vendas ----------------
dias["faixa_temperatura"] = pd.cut(dias["temperatura_media"], [-np.inf, 17, 20, 23, np.inf],
                                   labels=["Frio (<17°C)", "Ameno (17-20°C)",
                                           "Agradável (20-23°C)", "Quente (>23°C)"])

gold["clima_vendas"] = (dias.groupby(["condicao", "faixa_temperatura"], observed=True)
    .agg(dias=("data", "count"),
         media_itens=("itens_vendidos", "mean"),
         media_faturamento=("faturamento", "mean"))
    .reset_index())
display(gold["clima_vendas"])

print("Correlação com o número de itens vendidos no dia (de -1 a 1):")
display(dias[["itens_vendidos", "temperatura_media", "temperatura_max", "precipitacao_mm"]]
        .corr()["itens_vendidos"].drop("itens_vendidos").to_frame("correlação"))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
(dias.groupby("condicao")["itens_vendidos"].mean().sort_values()
     .plot.barh(ax=axes[0], color=COR, title="Média de itens vendidos por condição do tempo"))
axes[0].set_ylabel("")
axes[1].scatter(dias["temperatura_media"], dias["itens_vendidos"], color=COR, s=40, alpha=0.7,
                edgecolor="white", linewidth=1)
axes[1].set_title("Temperatura média × itens vendidos (cada ponto é um dia)")
axes[1].set_xlabel("°C")
axes[1].set_ylabel("itens")
plt.tight_layout()
plt.show()

# O que muda no "carrinho" conforme a temperatura: participação de cada categoria
ve["faixa_temperatura"] = pd.cut(ve["temperatura_media"], [-np.inf, 17, 20, 23, np.inf],
                                 labels=["Frio (<17°C)", "Ameno (17-20°C)",
                                         "Agradável (20-23°C)", "Quente (>23°C)"])
mix = (ve.groupby(["faixa_temperatura", "categoria"], observed=True)["quantidade"].sum()
         .unstack().pipe(lambda t: 100 * t.div(t.sum(axis=1), axis=0)))
print("Participação de cada categoria nas vendas (%), por faixa de temperatura:")
display(mix.round(1))

In [ ]:
# ---------------- 3.4 Desperdício por categoria de produto ----------------
pe = perdas.merge(cardapio[["id_produto", "nome_produto", "categoria"]], on="id_produto", how="left")
fat_cat = ve.groupby("categoria")["valor_total"].sum()

gold["desperdicio_categoria"] = (pe.groupby("categoria")
    .agg(registros=("id_perda", "count"),
         qtd_perdida=("quantidade", "sum"),
         prejuizo=("valor_prejuizo", "sum"))
    .join(fat_cat.rename("faturamento"))
    .reset_index())
gold["desperdicio_categoria"]["prejuizo_pct_faturamento"] = (100 * gold["desperdicio_categoria"]["prejuizo"]
                                                             / gold["desperdicio_categoria"]["faturamento"])
gold["desperdicio_categoria"] = gold["desperdicio_categoria"].sort_values("prejuizo", ascending=False)
display(gold["desperdicio_categoria"])

gold["desperdicio_produto_motivo"] = (pe.groupby(["categoria", "nome_produto", "motivo"])
    .agg(qtd_perdida=("quantidade", "sum"), prejuizo=("valor_prejuizo", "sum"))
    .reset_index().sort_values("prejuizo", ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
(gold["desperdicio_categoria"].set_index("categoria")["prejuizo"].sort_values()
     .plot.barh(ax=axes[0], color=COR, title="Prejuízo com perdas por categoria (R$)"))
axes[0].set_ylabel("")
(pe.groupby("nome_produto")["valor_prejuizo"].sum().nlargest(10).sort_values()
     .plot.barh(ax=axes[1], color=COR, title="Os 10 produtos com maior prejuízo (R$)"))
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# ---------------- 3.5 Turmas × dias de pico de venda ----------------
# Média de vendas por dia (e não o total), porque nem todo dia da semana aparece o mesmo número de vezes
n_dias_semana = dias.groupby("dia_semana")["data"].nunique()

tdp = (ve.dropna(subset=["turma"]).groupby(["turma", "dia_semana"])["id_venda"].nunique()
         .rename("vendas").reset_index())
tdp["media_vendas_por_dia"] = tdp["vendas"] / tdp["dia_semana"].map(n_dias_semana)
gold["turma_dia_pico"] = tdp

pico = (tdp.loc[tdp.groupby("turma")["media_vendas_por_dia"].idxmax(), ["turma", "dia_semana"]]
           .rename(columns={"dia_semana": "dia_de_pico"}).set_index("turma")
           .reindex([t for t in ORDEM_TURMAS if t in set(tdp["turma"])]))

mapa = (tdp.pivot(index="turma", columns="dia_semana", values="media_vendas_por_dia")
           .reindex(index=pico.index, columns=[d for d in ORDEM_DIAS if d in set(tdp["dia_semana"])]))
plt.figure(figsize=(9, 8))
sns.heatmap(mapa, cmap="Blues", annot=True, fmt=".1f", linewidths=0.5,
            cbar_kws={"label": "vendas por dia (média)"})
plt.title("Média de vendas por dia — turma × dia da semana")
plt.xlabel("")
plt.ylabel("")
plt.tight_layout()
plt.show()

print("Dia de pico de cada turma:")
display(pico.T)

mapa_hora = (ve.groupby(["dia_semana", "hora"])["id_venda"].nunique().unstack(fill_value=0)
               .div(n_dias_semana, axis=0)
               .reindex([d for d in ORDEM_DIAS if d in n_dias_semana.index]))
plt.figure(figsize=(12, 3.5))
sns.heatmap(mapa_hora, cmap="Blues", linewidths=0.5, cbar_kws={"label": "vendas (média)"})
plt.title("Horários de pico: média de vendas por dia da semana e hora")
plt.xlabel("hora")
plt.ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# ---------------- 3.6 Bônus: reajuste de preços × inflação ----------------
# Inflação acumulada de cada categoria: (1 + i1) x (1 + i2) x ... - 1
infl = (inflacao.sort_values("mes_referencia")
                .groupby("categoria_produto")["indice_inflacao"]
                .apply(lambda s: (np.prod(1 + s / 100) - 1) * 100)
                .rename("inflacao_acumulada_pct"))
print(f"Inflação publicada de {inflacao['mes_referencia'].min()} a {inflacao['mes_referencia'].max()}")

# Preço praticado de cada produto no 1º e no 2º semestre (mediana do valor unitário nas vendas)
ve["semestre"] = np.where(ve["data_hora"].dt.month <= 6, "1º semestre", "2º semestre")
precos = ve.groupby(["categoria", "nome_produto", "semestre"])["valor_unitario"].median().unstack()

if {"1º semestre", "2º semestre"} <= set(precos.columns):
    precos = precos.dropna()
    reajuste = (100 * (precos["2º semestre"] / precos["1º semestre"] - 1)).groupby("categoria").mean()
    gold["reajuste_vs_inflacao"] = (pd.concat([reajuste.rename("reajuste_medio_pct"), infl], axis=1)
                                      .rename_axis("categoria").reset_index())
    gold["reajuste_vs_inflacao"]["diferenca_pp"] = (gold["reajuste_vs_inflacao"]["reajuste_medio_pct"]
                                                    - gold["reajuste_vs_inflacao"]["inflacao_acumulada_pct"])
    display(gold["reajuste_vs_inflacao"])
    print("diferenca_pp negativa = o preço subiu MENOS que a inflação (a margem encolheu).")
else:
    gold["reajuste_vs_inflacao"] = infl.rename_axis("categoria").reset_index()
    print("Ainda não há vendas nos dois semestres; mostrando só a inflação acumulada.")
    display(gold["reajuste_vs_inflacao"])

## 4. Big Data com o ecossistema Hadoop (Aula 08)

Em Big Data, quando os dados não cabem em um computador, o trabalho é dividido entre várias máquinas.
O Hadoop organiza isso. Aqui aplicamos as três peças que fazem sentido para a cantina:

- **MapReduce** — divide uma contagem grande em pedaços (*map*), agrupa os pedaços iguais (*shuffle*)
  e soma tudo no final (*reduce*).
- **Sqoop** — leva tabelas para um banco de dados relacional (aqui, um arquivo SQLite).
- **Hive** — faz perguntas aos dados usando **SQL**.

### 4.1 MapReduce: produtos mais vendidos e faturamento por forma de pagamento

In [ ]:
# Etapa Map: transforma cada venda em um par (chave, valor)
def mapper(registros, chave, valor):
    pares = []
    for registro in registros:
        if pd.notna(registro[chave]):
            pares.append((registro[chave], registro[valor]))
    return pares


# Etapa Shuffle: junta todos os valores de uma mesma chave
def shuffle(pares):
    agrupado = {}
    for chave, valor in pares:
        agrupado.setdefault(chave, []).append(valor)
    return agrupado


# Etapa Reduce: soma os valores de cada chave
def reducer(agrupado):
    return {chave: sum(valores) for chave, valores in agrupado.items()}


def job_mapreduce(df, chave, valor, n_blocos=4):
    # Simula o HDFS: os dados são divididos em blocos, e cada bloco é mapeado separadamente
    registros = df[[chave, valor]].to_dict("records")
    tamanho = int(np.ceil(len(registros) / n_blocos))
    blocos = [registros[i:i + tamanho] for i in range(0, len(registros), tamanho)]
    pares = [par for bloco in blocos for par in mapper(bloco, chave, valor)]
    return reducer(shuffle(pares))


itens_por_produto = job_mapreduce(ve, "nome_produto", "quantidade")
print("[MapReduce] Top 10 produtos por itens vendidos:")
for produto, total in sorted(itens_por_produto.items(), key=lambda x: -x[1])[:10]:
    print(f"  {produto:<32} {total:>7,}")

fat_pagamento = job_mapreduce(ve, "forma_pagamento", "valor_total")
print("\n[MapReduce] Faturamento por forma de pagamento:")
for forma, total in sorted(fat_pagamento.items(), key=lambda x: -x[1]):
    print(f"  {forma:<20} R$ {total:>12,.2f}")

# Conferência: o MapReduce tem que bater com o groupby do pandas
confere = ve.groupby("nome_produto")["quantidade"].sum().to_dict()
print("\nMapReduce confere com o pandas?", confere == itens_por_produto)

### 4.2 Sqoop: exportando as tabelas para um banco de dados

O Sqoop transfere dados entre o Hadoop e bancos relacionais. Aqui levamos as tabelas limpas
e as tabelas Gold para um banco **SQLite** (`cantina.db`), que pode ser aberto por qualquer
ferramenta de BI ou planilha.

In [ ]:
def para_sql(df):
    # Bancos SQL não entendem o tipo "Period" do pandas nem categorias: viram texto
    df = df.copy()
    for coluna in df.columns:
        if isinstance(df[coluna].dtype, (pd.PeriodDtype, pd.CategoricalDtype)):
            df[coluna] = df[coluna].astype(str)
    return df


CAMINHO_BANCO = os.path.join(PASTA_SAIDA, "cantina.db")
if os.path.exists(CAMINHO_BANCO):
    os.remove(CAMINHO_BANCO)
conn = sqlite3.connect(CAMINHO_BANCO)

tabelas_banco = {"vendas_enriquecidas": ve, "devedores": dev, "perdas": pe, "cardapio": cardapio,
                 "alunos": alunos, "dias": dias, **{f"gold_{n}": t for n, t in gold.items()}}
for nome, tabela in tabelas_banco.items():
    para_sql(tabela).to_sql(nome, conn, index=False)

print("[Sqoop] Tabelas exportadas para", CAMINHO_BANCO)
display(pd.read_sql("SELECT name AS tabela FROM sqlite_master WHERE type = 'table'", conn))

### 4.3 Hive: perguntas em SQL

O Hive permite consultar dados gigantes com SQL, a linguagem padrão de bancos de dados.

In [ ]:
print("[Hive] Os 10 produtos que mais deram lucro:")
display(pd.read_sql('''
    SELECT nome_produto, categoria,
           SUM(quantidade)          AS itens,
           ROUND(SUM(valor_total), 2) AS faturamento,
           ROUND(SUM(lucro), 2)       AS lucro,
           ROUND(100.0 * SUM(lucro) / SUM(valor_total), 1) AS margem_pct
    FROM vendas_enriquecidas
    GROUP BY nome_produto, categoria
    ORDER BY lucro DESC
    LIMIT 10
''', conn))

print("[Hive] Ticket médio por segmento e turno:")
display(pd.read_sql('''
    SELECT segmento, turno,
           COUNT(DISTINCT id_venda) AS vendas,
           ROUND(SUM(valor_total) / COUNT(DISTINCT id_venda), 2) AS ticket_medio
    FROM vendas_enriquecidas
    WHERE segmento IS NOT NULL
    GROUP BY segmento, turno
    ORDER BY ticket_medio DESC
''', conn))

print("[Hive] Turmas com maior valor em atraso:")
display(pd.read_sql('''
    SELECT turma,
           COUNT(*) AS dividas,
           ROUND(SUM(valor_devido), 2)   AS total_fiado,
           ROUND(SUM(valor_atrasado), 2) AS total_atrasado
    FROM devedores
    WHERE turma IS NOT NULL
    GROUP BY turma
    ORDER BY total_atrasado DESC
    LIMIT 10
''', conn))

## 5. Regressão: quantos itens a cantina vai vender amanhã? (Aula 06)

**Problema real:** a cantina prepara salgados, lanches e sucos **antes** de saber quanto vai vender.
Se prepara demais, vira perda (seção 3.4); se prepara de menos, perde venda.

**Solução:** um modelo de **regressão** (que prevê um número) aprende com os dias anteriores
como o dia da semana, o clima e o calendário influenciam o volume de vendas.

- **Alvo (o que queremos prever):** `itens_vendidos` no dia.
- **Pistas (features):** dia da semana, dia do mês, temperatura média e máxima, chuva,
  condição do tempo, turno letivo e tipo de dia.
- **Modelo:** Random Forest — o mesmo da aula: várias "árvores de decisão" votando juntas.

Diferente da aula, separamos treino e teste **pela data** (treina com os dias mais antigos e
testa nos mais recentes). É assim que o modelo seria usado na vida real: prevendo o futuro.

In [ ]:
ALVO_REGRESSAO = "itens_vendidos"
num_reg = ["temperatura_media", "temperatura_max", "precipitacao_mm", "dia_do_mes"]
cat_reg = ["dia_semana", "condicao", "turno_letivo", "tipo_dia"]

base_reg = dias.sort_values("data").reset_index(drop=True)
X = base_reg[num_reg + cat_reg]
y = base_reg[ALVO_REGRESSAO]

# shuffle=False: os 20% de dias mais recentes ficam para o teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
print(f"Treino: {len(X_train)} dias | Teste: {len(X_test)} dias "
      f"(a partir de {base_reg.loc[X_test.index[0], 'data']:%d/%m/%Y})")


def criar_preprocessador(numericas, categoricas):
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())])
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    return ColumnTransformer(transformers=[
        ("num", numeric_transformer, numericas),
        ("cat", categorical_transformer, categoricas)])


modelo_reg = Pipeline(steps=[
    ("preprocessor", criar_preprocessador(num_reg, cat_reg)),
    ("regressor", RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=42))])
modelo_reg.fit(X_train, y_train)
y_pred = modelo_reg.predict(X_test)

# Comparação com um "chute" simples: prever sempre a média do treino
chute = np.full(len(y_test), y_train.mean())
metricas_reg = pd.DataFrame({
    "Modelo (Random Forest)": [mean_absolute_error(y_test, y_pred),
                               np.sqrt(mean_squared_error(y_test, y_pred)),
                               r2_score(y_test, y_pred)],
    "Chute (média)": [mean_absolute_error(y_test, chute),
                      np.sqrt(mean_squared_error(y_test, chute)),
                      r2_score(y_test, chute)]},
    index=["MAE (erro médio, em itens)", "RMSE", "R²"])
display(metricas_reg)
print(f"Em média, o modelo erra {metricas_reg.iloc[0, 0]:.0f} itens por dia "
      f"(o chute erraria {metricas_reg.iloc[0, 1]:.0f}). "
      f"Média de itens vendidos por dia: {y.mean():.0f}.")

In [ ]:
datas_teste = base_reg.loc[X_test.index, "data"]
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(datas_teste, y_test.values, color=CORES[0], linewidth=2, marker="o", markersize=5, label="Real")
ax.plot(datas_teste, y_pred, color=CORES[1], linewidth=2, marker="o", markersize=5, label="Previsto")
ax.set_title("Itens vendidos por dia no período de teste: real × previsto")
ax.set_ylabel("itens")
ax.legend()
plt.tight_layout()
plt.show()

# O que mais pesou na previsão
nomes = modelo_reg.named_steps["preprocessor"].get_feature_names_out()
importancia = (pd.Series(modelo_reg.named_steps["regressor"].feature_importances_, index=nomes)
                 .rename(lambda n: n.split("__", 1)[1]).nlargest(12).sort_values())
importancia.plot.barh(color=COR, figsize=(9, 5), title="O que mais influencia o volume de vendas")
plt.xlabel("importância")
plt.tight_layout()
plt.show()

### 5.1 Usando o modelo: previsão e sugestão de produção

Informe como será o dia (pela previsão do tempo) e o modelo estima o total de itens.
Com a participação histórica de cada produto, sugerimos quanto preparar dos **perecíveis**.

In [ ]:
novo_dia = pd.DataFrame([{
    "temperatura_media": 27.0,
    "temperatura_max": 32.0,
    "precipitacao_mm": 0.0,
    "dia_do_mes": 3,
    "dia_semana": "Sex",
    "condicao": "ensolarado",
    "turno_letivo": "integral",
    "tipo_dia": "letivo",
}])
previsao_itens = modelo_reg.predict(novo_dia)[0]
print(f"Previsão para o dia informado: {previsao_itens:.0f} itens")

# Participação de cada produto, calculada só com dias de temperatura parecida (±3 °C)
parecidos = ve[(ve["temperatura_media"] - novo_dia.loc[0, "temperatura_media"]).abs() <= 3]
if parecidos.empty:
    parecidos = ve
participacao = parecidos.groupby(["categoria", "nome_produto"])["quantidade"].sum()
participacao = participacao / participacao.sum()
sugestao = (participacao * previsao_itens).round().astype(int).rename("sugestao_de_producao")
display(sugestao.sort_values(ascending=False).head(15).reset_index())

## 6. Classificação: esta compra fiado vai atrasar? (Aula 05)

**Problema real:** parte das vendas é "fiado" — o responsável paga depois. Algumas dívidas atrasam.

**Solução:** um modelo de **classificação** (que prevê uma categoria) aprende com as dívidas antigas
quais características aparecem mais nas que atrasaram.

- **Alvo:** `atrasou` = 1 se a dívida passou de **30 dias** sem pagamento (ou foi paga depois disso), 0 se foi paga em dia.
- **Pistas:** valor da dívida, quantas dívidas o aluno já tinha antes e quanto somavam, série, idade,
  segmento, turno, restrição alimentar, dia da semana e dia do mês.
- Usamos apenas dívidas com **mais de 30 dias**: as mais novas ainda não tiveram tempo de atrasar,
  então não sabemos o "final da história" delas.
- `class_weight="balanced"`: como atrasos são minoria, o modelo dá mais peso a eles para não ignorá-los.

In [ ]:
PRAZO_DIAS = 30

clf_base = dev.dropna(subset=["turma"]).sort_values(["ano_letivo", "id_aluno", "data_divida"]).copy()
# Histórico do aluno ATÉ aquela dívida (nunca usa informação do futuro)
clf_base["dividas_anteriores"] = clf_base.groupby(["ano_letivo", "id_aluno"]).cumcount()
clf_base["valor_anterior"] = (clf_base.groupby(["ano_letivo", "id_aluno"])["valor_devido"].cumsum()
                              - clf_base["valor_devido"])
clf_base["dia_semana"] = clf_base["data_divida"].dt.dayofweek.map(DIAS_SEMANA)
clf_base["dia_do_mes"] = clf_base["data_divida"].dt.day

data_ref = devedores["data_divida"].max()
clf_base = clf_base[clf_base["data_divida"] <= data_ref - pd.Timedelta(days=PRAZO_DIAS)]
dias_para_pagar = (clf_base["data_quitacao"] - clf_base["data_divida"]).dt.days
clf_base["atrasou"] = ((clf_base["status"] == STATUS_ATRASADO)
                       | (dias_para_pagar > PRAZO_DIAS)
                       | ((clf_base["status"] == STATUS_ABERTO))).astype(int)

num_clf = ["valor_devido", "dividas_anteriores", "valor_anterior", "serie", "idade", "dia_do_mes"]
cat_clf = ["segmento", "turno", "restricao_alimentar", "dia_semana"]
X = clf_base[num_clf + cat_clf]
y = clf_base["atrasou"]
print(f"{len(clf_base)} dívidas analisadas | {y.mean():.1%} atrasaram")

In [ ]:
if y.nunique() < 2:
    print("Ainda não há dívidas atrasadas E pagas em dia suficientes para treinar o modelo.")
else:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    modelo_clf = Pipeline(steps=[
        ("preprocessor", criar_preprocessador(num_clf, cat_clf)),
        ("classifier", RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                              class_weight="balanced", random_state=42))])
    modelo_clf.fit(X_train, y_train)
    y_pred = modelo_clf.predict(X_test)

    print(f"Acurácia: {accuracy_score(y_test, y_pred):.2%}  (acertos no geral)")
    print(f"Precisão: {precision_score(y_test, y_pred, average='macro'):.2%}  "
          "(quando o modelo diz 'vai atrasar', quanto acerta)")
    print(f"Recall:   {recall_score(y_test, y_pred, average='macro'):.2%}  "
          "(dos atrasos reais, quantos o modelo encontrou)")
    print(f"F1-Score: {f1_score(y_test, y_pred, average='macro'):.2%}  (equilíbrio entre os dois)")

    cm = confusion_matrix(y_test, y_pred)
    classes = ["Pagou em dia", "Atrasou"]
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
    plt.title("Matriz de confusão")
    plt.xlabel("Previsão do modelo")
    plt.ylabel("O que aconteceu")
    plt.tight_layout()
    plt.show()

### 6.1 Usando o modelo: risco de uma nova compra fiado

In [ ]:
if y.nunique() >= 2:
    nova_divida = X.iloc[[0]].copy()
    nova_divida["valor_devido"] = 18.50
    nova_divida["dividas_anteriores"] = 6
    nova_divida["valor_anterior"] = 75.00

    print("Dados da nova compra fiado:")
    display(nova_divida.T.rename(columns={nova_divida.index[0]: "valor"}))
    probabilidade = modelo_clf.predict_proba(nova_divida)[0]
    for classe, prob in zip(["Pagar em dia", "Atrasar"], probabilidade):
        print(f"Chance de {classe}: {prob:.1%}")

## 7. Apache Spark: SQL e streaming (Aula 09)

O **Spark** processa dados distribuídos entre várias máquinas e também dados que **chegam
continuamente** (*Structured Streaming*).

Na cantina, as vendas chegam em **lotes** ao longo do dia (é exatamente o que o gerador de dados faz).
Vamos:
1. usar o **Spark SQL** para recalcular uma tabela Gold e conferir com o pandas;
2. simular a chegada das vendas **dia após dia** e ver um painel sendo atualizado sozinho.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = SparkSession.builder.appName("CantinaEscolar").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark iniciado. Versão:", spark.version)

colunas_spark = ["id_venda", "data", "hora", "turma", "categoria", "nome_produto",
                 "quantidade", "valor_total", "forma_pagamento"]
ve_spark = ve[colunas_spark].copy()
ve_spark["data"] = ve_spark["data"].dt.strftime("%Y-%m-%d")
ve_spark[["turma", "categoria", "nome_produto"]] = (ve_spark[["turma", "categoria", "nome_produto"]]
                                                    .fillna("Não identificado"))
sdf = spark.createDataFrame(ve_spark)
sdf.createOrReplaceTempView("vendas")

print("[Spark SQL] Faturamento por categoria e mês:")
spark.sql('''
    SELECT substr(data, 1, 7)          AS mes,
           categoria,
           SUM(quantidade)             AS itens,
           ROUND(SUM(valor_total), 2)  AS faturamento
    FROM vendas
    GROUP BY substr(data, 1, 7), categoria
    ORDER BY mes, faturamento DESC
''').show(15, truncate=False)

total_spark = sdf.agg(F.sum("valor_total")).first()[0]
print(f"Faturamento total — Spark: R$ {total_spark:,.2f} | pandas: R$ {ve['valor_total'].sum():,.2f}")

### 7.1 Structured Streaming: painel ao vivo

Cada "dia" de vendas é gravado como um novo arquivo JSON numa pasta. O Spark vigia essa pasta
(`readStream`) e, a cada arquivo novo, **atualiza sozinho** o faturamento por categoria
(`outputMode("complete")` = sempre a tabela inteira, já somada).

In [ ]:
N_DIAS_SIMULADOS = 10

PASTA_STREAM = "/content/stream_cantina"
PASTA_ENTRADA = f"{PASTA_STREAM}/entrada"
PASTA_TEMP = f"{PASTA_STREAM}/temporarios"
PASTA_CHECKPOINT = f"{PASTA_STREAM}/checkpoint"
shutil.rmtree(PASTA_STREAM, ignore_errors=True)
os.makedirs(PASTA_ENTRADA)
os.makedirs(PASTA_TEMP)

esquema = StructType([
    StructField("data", StringType()),
    StructField("hora", IntegerType()),
    StructField("categoria", StringType()),
    StructField("quantidade", IntegerType()),
    StructField("valor_total", DoubleType()),
])
fluxo = spark.readStream.schema(esquema).json(PASTA_ENTRADA)

painel = (fluxo.groupBy("categoria")
               .agg(F.sum("quantidade").alias("itens"),
                    F.round(F.sum("valor_total"), 2).alias("faturamento")))

consulta = (painel.writeStream
                  .format("memory")
                  .queryName("painel_ao_vivo")
                  .outputMode("complete")
                  .option("checkpointLocation", PASTA_CHECKPOINT)
                  .start())

dias_simulados = sorted(ve_spark["data"].unique())[:N_DIAS_SIMULADOS]
for numero, dia in enumerate(dias_simulados, start=1):
    lote = ve_spark.loc[ve_spark["data"] == dia, ["data", "hora", "categoria", "quantidade", "valor_total"]]
    arquivo = f"lote_{numero:03d}.json"
    # Escreve na pasta temporária e só depois move: o Spark nunca lê um arquivo pela metade
    lote.to_json(os.path.join(PASTA_TEMP, arquivo), orient="records", lines=True, force_ascii=False)
    os.replace(os.path.join(PASTA_TEMP, arquivo), os.path.join(PASTA_ENTRADA, arquivo))
    consulta.processAllAvailable()   # espera o Spark processar o lote que acabou de chegar
    acumulado = spark.sql("SELECT SUM(itens) AS itens, SUM(faturamento) AS fat FROM painel_ao_vivo").first()
    print(f"Lote {numero:2d} ({dia}) chegou → acumulado: {acumulado['itens']:>6,} itens | "
          f"R$ {acumulado['fat']:>10,.2f}")

print("\nPainel após todos os lotes:")
painel_final = spark.sql("SELECT * FROM painel_ao_vivo ORDER BY faturamento DESC").toPandas()
display(painel_final)

consulta.stop()

In [ ]:
painel_final.set_index("categoria")["faturamento"].sort_values().plot.barh(
    color=COR, figsize=(9, 4), title=f"Faturamento por categoria nos {N_DIAS_SIMULADOS} dias do streaming (R$)")
plt.ylabel("")
plt.tight_layout()
plt.show()
spark.stop()

## 8. Resumo dos achados e exportação

Os principais números, calculados automaticamente a partir dos dados.

In [ ]:
fat_total = ve["valor_total"].sum()
lucro_total = ve["lucro"].sum()
prejuizo_perdas = pe["valor_prejuizo"].sum()
top_produto = ve.groupby("nome_produto")["quantidade"].sum().idxmax()
dia_pico = (ve.groupby("dia_semana")["id_venda"].nunique() / n_dias_semana).idxmax()
hora_pico = ve.groupby("hora")["id_venda"].nunique().idxmax()
cat_perda = gold["desperdicio_categoria"].iloc[0]
clima_top = dias.groupby("condicao")["itens_vendidos"].mean()
pct_fiado_atrasado = 100 * dev["valor_atrasado"].sum() / dev["valor_devido"].sum()

resumo = [
    ("Faturamento no período", f"R$ {fat_total:,.2f}"),
    ("Lucro bruto (venda - custo)", f"R$ {lucro_total:,.2f} ({100 * lucro_total / fat_total:.1f}%)"),
    ("Prejuízo com perdas", f"R$ {prejuizo_perdas:,.2f} ({100 * prejuizo_perdas / fat_total:.1f}% do faturamento)"),
    ("Produto mais vendido", top_produto),
    ("Dia da semana de pico", dia_pico),
    ("Horário de pico", f"{hora_pico}h"),
    ("Categoria com mais desperdício", f"{cat_perda['categoria']} (R$ {cat_perda['prejuizo']:,.2f})"),
    ("Tempo com mais vendas / menos vendas", f"{clima_top.idxmax()} / {clima_top.idxmin()}"),
    ("Fiado em atraso", f"{pct_fiado_atrasado:.1f}% do valor vendido fiado"),
    ("Erro médio da previsão de vendas", f"{metricas_reg.iloc[0, 0]:.0f} itens/dia "
                                         f"(média de {dias['itens_vendidos'].mean():.0f} itens/dia)"),
]
display(pd.DataFrame(resumo, columns=["Indicador", "Valor"]).set_index("Indicador"))

In [ ]:
# Salva cada tabela Gold em CSV (o banco cantina.db já foi salvo na seção 4.2)
for nome, tabela in gold.items():
    para_sql(tabela).to_csv(os.path.join(PASTA_SAIDA, f"gold_{nome}.csv"), index=False, encoding="utf-8-sig")
conn.close()

arquivo_zip = shutil.make_archive("/content/resultados_cantina", "zip", PASTA_SAIDA)
print("Arquivos gerados:", sorted(os.listdir(PASTA_SAIDA)))
print("Tudo compactado em:", arquivo_zip)

BAIXAR_RESULTADOS = False   # mude para True para baixar o .zip pelo navegador
if BAIXAR_RESULTADOS:
    from google.colab import files
    files.download(arquivo_zip)